In [ ]:
import os
os.environ["GROQ_API_KEY"] = "YOUR_GROQ_API_KEY"

In [ ]:
pip -q install langchain langchain_community langchain_openai faiss-cpu pypdf sentence-transformers

In [ ]:
from google.colab import files
num_files=int(input("how many PDF Files do you want to upload?:"))
print(f"Please select {num_files} PDF files to upload:")
uploaded=files.upload()
pdf_files=[f for f in uploaded.keys() if f.lower().endswith('.pdf')]
if len(pdf_files)!=num_files:
  print(f"Error:Expected{num_files}PDF files,but received {len(pdf_files)}")
else:
  print("\nUploaded PDF's")
  for i,pdf in enumerate(pdf_files,start=1):
    print(f"{i}.{pdf}")


In [ ]:
from langchain_community.document_loaders import PyPDFLoader
all_documents=[]
for pdf_file_name in pdf_files:
  print(f"Loading {pdf_file_name}...")
  loader=PyPDFLoader(pdf_file_name)
  documents=loader.load()
  all_documents.extend(documents)
  print("Total Pages Loaded:",len(all_documents))
  documents=all_documents

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
text_splitter=RecursiveCharacterTextSplitter(chunk_size=1000,chunk_overlap=500)
docs=text_splitter.split_documents(documents)
print("Total Chunks:",len(docs))

In [ ]:
from langchain_community.embeddings import HuggingFaceEmbeddings
embeddings=HuggingFaceEmbeddings(model_name="BAAI/bge-small-en-v1.5")

In [ ]:
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
embeddings=HuggingFaceEmbeddings(model_name="BAAI/bge-small-en-v1.5")
vectorstores=FAISS.from_documents(docs,embeddings)
print("Vector DB Created Successfully")
print(vectorstores)

In [ ]:
retriever=vectorstores.as_retriever(search_kwargs={"k":5})
print(retriever)

In [ ]:
print(vectorstores.index_to_docstore_id)

In [ ]:
from langchain_openai import ChatOpenAI
import os

llm = ChatOpenAI(
    model="llama-3.3-70b-versatile",
    api_key=os.getenv("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1"
)

In [ ]:
!pip -q install flask pyngrok

In [ ]:
from flask import Flask, request, jsonify, render_template_string
from pyngrok import ngrok
ngrok.kill()

In [ ]:
app = Flask(__name__)

In [ ]:
def financial_analyst(question):

    prompt = f"""
You are an expert AI Financial Analyst.

Rules:
- If the PDF contains tabular financial information relevant to the question,
  return the answer as a valid HTML table.
- Use bullet points for lists.
- Use paragraphs for summaries.
- Return valid HTML.

Question:
{question}
"""

    response = rag_chain.invoke(prompt)

    return str(response)

In [ ]:
HTML = """
<!DOCTYPE html>
<html>
<head>

<title>AI Financial Analyst</title>

<style>

*{
    margin:0;
    padding:0;
    box-sizing:border-box;
    font-family:Arial,sans-serif;
}

body{
    display:flex;
    height:100vh;
    background:#212121;
    color:white;
}

/* Sidebar */

.sidebar{
    width:260px;
    background:#171717;
    padding:20px;
    overflow-y:auto;
    border-right:1px solid #333;
}

.sidebar h2{
    margin-bottom:20px;
}

.new-chat{
    width:100%;
    padding:12px;
    border:none;
    border-radius:10px;
    background:#19c37d;
    cursor:pointer;
    font-weight:bold;
}

.sidebar h3{
    margin-top:25px;
    margin-bottom:15px;
}

/* Chat History */

#history{
    max-height:500px;
    overflow-y:auto;
}

.history-item{
    background:#2d2d2d;
    padding:12px;
    margin-bottom:10px;
    border-radius:8px;
    cursor:pointer;
    word-wrap:break-word;
}

.history-item:hover{
    background:#3d3d3d;
}

/* Main Area */

.main{
    flex:1;
    display:flex;
    flex-direction:column;
}

.header{
    padding:20px;
    border-bottom:1px solid #333;
    font-size:24px;
    font-weight:bold;
}

.chat-container{
    flex:1;
    overflow-y:auto;
    padding:30px;
}

/* Messages */

.message{
    display:flex;
    margin-bottom:20px;
}

.user{
    justify-content:flex-end;
}

.bot{
    justify-content:flex-start;
}

.bubble{
    max-width:75%;
    padding:15px;
    border-radius:15px;
    line-height:1.6;
}

.user .bubble{
    background:#19c37d;
    color:black;
}

.bot .bubble{
    background:#2d2d2d;
    color:white;
}

/* Typing Indicator */

.typing{
    color:#bdbdbd;
    margin-bottom:15px;
}

/* Input Area */

.input-area{
    display:flex;
    gap:10px;
    padding:20px;
    border-top:1px solid #333;
}

textarea{
    flex:1;
    resize:none;
    height:55px;
    border:none;
    border-radius:12px;
    padding:15px;
    background:#2d2d2d;
    color:white;
    font-size:15px;
}

button{
    padding:0 25px;
    border:none;
    border-radius:12px;
    background:#19c37d;
    cursor:pointer;
    font-weight:bold;
}

/* TABLE STYLING */

table{
    width:100%;
    border-collapse:collapse;
    margin-top:15px;
    margin-bottom:15px;
    background:#2d2d2d;
    border-radius:12px;
    overflow:hidden;
}

th{
    background:#19c37d;
    color:black;
    padding:12px;
    text-align:left;
    font-weight:bold;
}

td{
    padding:12px;
    border-bottom:1px solid #444;
}

tr:nth-child(even){
    background:#262626;
}

tr:hover{
    background:#3a3a3a;
}

/* Bullet Lists */

ul{
    margin-left:20px;
    margin-top:10px;
}

li{
    margin-bottom:6px;
}

</style>

</head>

<body>

<div class="sidebar">

    <h2>📊 AI Financial Analyst</h2>

    <button class="new-chat"
            onclick="newChat()">
        + New Chat
    </button>

    <h3>Chat History</h3>

    <div id="history"></div>

</div>

<div class="main">

    <div class="header">
        AI Financial Analyst
    </div>

    <div class="chat-container" id="chat">

        <div class="message bot">

            <div class="bubble">

                Hello! Upload a financial report and ask me anything.

                <br><br>

                Example questions:
                <ul>
                    <li>Summarize the annual report.</li>
                    <li>Show revenue and profit trends.</li>
                    <li>List major risk factors.</li>
                    <li>Display balance sheet data.</li>
                </ul>

            </div>

        </div>

    </div>

    <div class="input-area">

        <textarea id="question"
                  placeholder="Ask anything about the financial report..."></textarea>

        <button onclick="askQuestion()">
            Send
        </button>

    </div>

</div>

<script>
/* VARIABLES */

let conversations =
JSON.parse(localStorage.getItem("conversations")) || [];

let currentConversation = [];


/* ASK QUESTION */

function askQuestion(){

    let question =
    document.getElementById("question").value.trim();

    if(question === "") return;

    let chat =
    document.getElementById("chat");

    /* Show User Message */

    chat.innerHTML += `
    <div class="message user">
        <div class="bubble">
            ${question}
        </div>
    </div>`;

    /* Typing Indicator */

    chat.innerHTML += `
    <div class="typing" id="typing">
        AI Financial Analyst is thinking...
    </div>`;

    chat.scrollTop = chat.scrollHeight;

    fetch('/ask',{

        method:'POST',

        headers:{
            'Content-Type':
            'application/x-www-form-urlencoded'
        },

        body:'question=' +
        encodeURIComponent(question)

    })

    .then(response => response.json())

    .then(data => {

        /* Remove Typing */

        let typing =
        document.getElementById("typing");

        if(typing){
            typing.remove();
        }

        /* Show AI Response */

        chat.innerHTML += `
        <div class="message bot">
            <div class="bubble">
                ${data.answer}
            </div>
        </div>`;

        /* Save Conversation */

        currentConversation.push({

            question: question,

            answer: data.answer

        });

        document.getElementById("question").value = "";

        chat.scrollTop = chat.scrollHeight;

    })

    .catch(error => {

        let typing =
        document.getElementById("typing");

        if(typing){
            typing.remove();
        }

        chat.innerHTML += `
        <div class="message bot">
            <div class="bubble">
                Error: Unable to get response.
            </div>
        </div>`;

    });

}


/* NEW CHAT */

function newChat(){

    if(currentConversation.length > 0){

        conversations.push({

            title:
            currentConversation[0].question,

            messages:
            [...currentConversation]

        });

        localStorage.setItem(

            "conversations",

            JSON.stringify(conversations)

        );
    }

    currentConversation = [];

    updateSidebar();

    document.getElementById("chat").innerHTML = `

    <div class="message bot">

        <div class="bubble">

            New chat started.

            <br><br>

            Ask me anything about your financial reports.

        </div>

    </div>`;
}


/* UPDATE SIDEBAR */

function updateSidebar(){

    let history =
    document.getElementById("history");

    history.innerHTML = "";

    conversations.forEach((chat,index)=>{

        history.innerHTML += `

        <div class="history-item"

             onclick="openChat(${index})">

            ${chat.title}

        </div>`;

    });

}


/* OPEN PREVIOUS CHAT */

function openChat(index){

    let chatBox =
    document.getElementById("chat");

    chatBox.innerHTML = "";

    conversations[index].messages.forEach(msg=>{

        chatBox.innerHTML += `

        <div class="message user">

            <div class="bubble">

                ${msg.question}

            </div>

        </div>

        <div class="message bot">

            <div class="bubble">

                ${msg.answer}

            </div>

        </div>`;
    });

    currentConversation =
    [...conversations[index].messages];

    chatBox.scrollTop =
    chatBox.scrollHeight;
}


/* SEND MESSAGE USING ENTER */

document.getElementById("question")

.addEventListener("keypress",

function(event){

    if(event.key === "Enter" && !event.shiftKey){

        event.preventDefault();

        askQuestion();
    }

});


/* LOAD HISTORY */

updateSidebar();

</script>

</body>

</html>
"""

In [ ]:
@app.route('/')
def home():
    return render_template_string(HTML)


@app.route('/ask', methods=['POST'])
def ask():

    question = request.form['question']

    answer = financial_analyst(question)

    return jsonify({
        "answer": answer
    })

In [ ]:
ngrok.set_auth_token( "YOUR_GROQ_API_KEY")

In [ ]:
public_url = ngrok.connect(5000)

print(public_url)

In [ ]:
app.run(
    host='0.0.0.0',
    port=5000,
    use_reloader=False
)